In [ ]:
import os
import subprocess
from pathlib import Path

# ==== 可手动修改的参数 ====
ROOT_FOLDER = Path(r"")  # 所有视频文件夹的上级目录
VIDEO_EXTENSIONS = ['.mp4', '.mkv', '.mov', '.avi', '.flv']

def get_video_duration(file_path):
    """获取视频总时长（单位：秒）"""
    try:
        result = subprocess.run(
            [
                'ffprobe', '-v', 'error',
                '-show_entries', 'format=duration',
                '-of', 'default=noprint_wrappers=1:nokey=1',
                str(file_path)
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        return float(result.stdout.strip())
    except Exception as e:
        print(f"⚠️ 获取时长失败：{file_path}，原因：{e}")
        return 0

def split_video(file_path, duration, parts):
    """使用 ffmpeg 分割视频为指定数量的部分"""
    parent = file_path.parent
    name = file_path.stem
    ext = file_path.suffix
    segment_duration = duration / parts

    for i in range(parts):
        start = int(segment_duration * i)
        output_path = parent / f"{name}_part{i+1}{ext}"
        cmd = [
            "ffmpeg", "-y", "-i", str(file_path),
            "-ss", str(start),
            "-t", str(int(segment_duration)),
            "-c", "copy", str(output_path)
        ]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print(f"✅ 分割完成：{output_path.name}")

def move_original(file_path, target_folder):
    target_folder.mkdir(parents=True, exist_ok=True)
    file_path.rename(target_folder / file_path.name)
    print(f"📦 原始大文件已移动到：{target_folder / file_path.name}")

def is_video_file(path):
    return path.suffix.lower() in VIDEO_EXTENSIONS

def process_folder(root_folder):
    archive_folder = root_folder / "long_videos"

    for subdir in root_folder.iterdir():
        if subdir.is_dir() and subdir.name != "long_videos":
            for file in subdir.iterdir():
                if file.is_file() and is_video_file(file):
                    size_gb = file.stat().st_size / 1024**3

                    # 判断分段数量
                    if 2 <= size_gb < 4:
                        parts = 2
                    elif 4 <= size_gb < 6:
                        parts = 3
                    elif size_gb >= 6:
                        print(f"⚠️ 文件过大（{size_gb:.2f} GB），暂不处理：{file}")
                        continue
                    else:
                        continue  # 小于 2GB 的不处理

                    print(f"\n📍 准备分割（{size_gb:.2f} GB, {parts} 段）：{file}")
                    duration = get_video_duration(file)
                    if duration > 0:
                        split_video(file, duration, parts)
                        move_original(file, archive_folder)

if __name__ == "__main__":
    process_folder(ROOT_FOLDER)
